# Reinforcement Learning — Module 1 Foundations

## Learning Objectives
- Understand the core RL framework: agent, environment, state, action, reward
- Learn Markov Decision Processes (MDPs) as the mathematical foundation of RL
- Implement Q-learning, a model-free RL algorithm, from scratch
- Experiment with OpenAI Gym environments

## Why This Matters
Skillsoft includes 1h 03m of RL covering essential components (agent, action, environment), reward hypothesis, environment types, OpenAI Gym/TensorFlow Agents installation, Q-learning, and Deep Q-learning. RL is the foundation for decision-making AI systems and is increasingly important for ML Architect roles involving autonomous systems.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Try to import OpenAI Gym; if not available, create a simple MDP environment
try:
    import gym
    GYM_AVAILABLE = True
    print("OpenAI Gym available")
except ImportError:
    GYM_AVAILABLE = False
    print("OpenAI Gym not installed. Creating simple MDP environment instead.")

sns.set_style("whitegrid")
np.random.seed(42)
%matplotlib inline


## 1. The RL Framework: Agent-Environment Interaction


In [ ]:
# The core RL loop
# At each time step t:
# 1. Agent observes state S_t
# 2. Agent selects action A_t using policy π(A_t|S_t)
# 3. Environment transitions to S_{t+1} and produces reward R_{t+1}
# 4. Agent updates policy based on (S_t, A_t, R_{t+1}, S_{t+1})

# Markov Decision Process (MDP) tuple: (S, A, P, R, γ, ρ_0)
# S: set of states
# A: set of actions
# P(s'|s, a): transition probability
# R(s, a, s'): reward function
# γ: discount factor (0 ≤ γ ≤ 1)
# ρ_0: initial state distribution

print("RL Framework Components")
print("=" * 40)
print("  • Agent: Learns a policy π(a|s) to select actions")
print("  • Environment: Provides transitions and rewards")
print("  • State (S): What the agent observes")
print("  • Action (A): What the agent does")
print("  • Reward (R): Scalar feedback from environment")
print("  • Policy (π): Mapping from states to action probabilities")
print("  • Value Function (V, Q): Expected cumulative reward")
print("  • Q-learning: Learning the action-value function")


## 2. Simple MDP: Grid World Navigation

# A 4x4 grid where the agent starts at (0,0) and goal is at (3,3)
# Actions: 0=up, 1=right, 2=down, 3=left
# Reward: -1 per step, +10 at goal, -10 at trap


In [ ]:
# Simple Grid World MDP
class GridWorldMDP:
    def __init__(self, size=4, trap_pos=None, goal_pos=None):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4  # up, right, down, left

# Default: trap at (1,1), goal at (3,3)
        if trap_pos is None:
            self.trap_state = 1 * size + 1  # state index
        else:
            self.trap_state = trap_pos[0] * size + trap_pos[1]
        if goal_pos is None:
            self.goal_state = (size-1) * size + (size-1)  # bottom-right
        else:
            self.goal_state = goal_pos[0] * size + goal_pos[1]

        # States: 0 to size^2-1, indexed row-major
        # Transitions: deterministic
        self.transitions = {}
        self.rewards = {}

        for s in range(self.n_states):
            r, c = divmod(s, size)
            self.transitions[s] = {}
            self.rewards[s] = {}

            for action, (dr, dc) in enumerate([(-1, 0), (0, 1), (1, 0), (0, -1)]):
                nr, nc = r + dr, c + dc
                # Stay in place if would move off grid
                if nr < 0 or nr >= size or nc < 0 or nc >= size:
                    nr, nc = r, c
                n_state = nr * size + nc
                self.transitions[s][action] = n_state

                # Reward: -1 per step, +10 at goal, -10 at trap
                if n_state == self.goal_state:
                    self.rewards[s][action] = 10
                elif n_state == self.trap_state:
                    self.rewards[s][action] = -10
                else:
                    self.rewards[s][action] = -1

        self.goal_state = self.goal_state
        self.trap_state = self.trap_state

    def step(self, state, action):
        next_state = self.transitions[state][action]
        reward = self.rewards[state][action]
        done = (next_state == self.goal_state or next_state == self.trap_state)
        return next_state, reward, done

    def reset(self):
        return 0  # Start at top-left state 0

    def render(self, state=None, policy=None):
        if state is None:
            state = self.goal_state  # Just show goal for now
        size = self.size
        grid = [['.' for _ in range(size)] for _ in range(size)]
        grid[self.trap_state // size][self.trap_state % size] = 'T'
        grid[self.goal_state // size][self.goal_state % size] = 'G'
        r, c = divmod(state, size)
        grid[r][c] = 'A'
        for row in grid:
            print(' '.join(row))
        print()

env = GridWorldMDP(size=4)
print("Grid World (T=Trap, G=Goal, A=Agent position):")
env.render()

print("Actions: 0=Up, 1=Right, 2=Down, 3=Left")
print("Reward: -1 per step, +10 at Goal, -10 at Trap")


## 3. Q-Learning from Scratch

Q-learning update: Q(s,a) ← Q(s,a) + α [r + γ max_a' Q(s',a') - Q(s,a)]

# Q-table: Q[state, action] → expected cumulative reward
# α: learning rate (0 = no learning, 1 = learn fully from single experience)
# γ: discount factor (future rewards counted less if γ < 1)


In [ ]:
# Q-Learning implementation
def q_learning(env, n_episodes=1000, alpha=0.1, gamma=0.99, epsilon=0.1):
    """Q-learning algorithm.

    env: GridWorldMDP environment
    n_episodes: number of training episodes
    alpha: learning rate
    gamma: discount factor
    epsilon: exploration rate (ε-greedy policy)
    """
    # Initialize Q-table with zeros
    Q = np.zeros((env.n_states, env.n_actions))

    for episode in range(n_episodes):
        state = env.reset()
        done = False
        total_reward = 0

        while not done:
            # ε-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(env.n_actions)  # explore
            else:
                action = np.argmax(Q[state, :])  # exploit

            # Take action, observe reward and next state
            next_state, reward, done = env.step(state, action)

            # Q-learning update
            best_next_q = np.max(Q[next_state, :]) if not done else 0
            td_target = reward + gamma * best_next_q
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            state = next_state
            total_reward += reward

            if done:
                break

    return Q

# Train the agent
env = GridWorldMDP(size=4)
Q_table = q_learning(env, n_episodes=500, alpha=0.1, gamma=0.99, epsilon=0.1)

print("Q-Learning Results:")
print(f"Trained Q-table shape: {Q_table.shape}")
print("Learned Q-values (higher = better):")
print(Q_table)

# Evaluate: see if agent can reach goal from start
state = env.reset()
done = False
steps = 0
policy_actions = {0: 'Up', 1: 'Right', 2: 'Down', 3: 'Left'}
path = [state]

while not done and steps < 50:
    action = np.argmax(Q_table[state, :])
    state, reward, done = env.step(state, action)
    path.append(state)

print(f"\nEvaluated path from start to termination:")
print(f"  Path states: {path}")
print(f"  Total steps: {steps + 1}")
print(f"  Total reward: {reward}")
print(f"  Reached goal: {state == env.goal_state}")


## 4. OpenAI Gym: Cart-Pole Example (if available)


In [ ]:
if GYM_AVAILABLE:
    # Cart-Pole: balance a pole on a moving cart
    env = gym.make('CartPole-v1', render_mode='human')

    # Simple policy: always move toward pole
    n_episodes = 3
    for episode in range(n_episodes):
        state = env.reset()
        done = False
        total_reward = 0
        while not done:
            # Simple heuristic: move direction of pole
            pole_angle = state[2]
            action = 1 if pole_angle > 0 else 0  # 1=right, 0=left
            state, reward, done, info = env.step(action)
            total_reward += reward
        print(f"CartPole episode {episode+1}: total reward = {total_reward}")
    env.close()

else:
    print("Skipping OpenAI Gym demo (not installed)")
    print("To install: pip install gym")


## 5. Exercises


### Exercise 1: Grid World with Different Parameters
Reset the grid world with the goal at (0,0) (top-left) and evaluate if Q-learning learns to go there.


### Exercise 2: Q-Learning with Different Learning Rates
Compare Q-learning with α=0.01 (slow learning) vs α=0.5 (fast learning) vs α=1.0 (no smoothing). Track how many episodes each needs to reach the goal.


### Exercise 3: Effect of Discount Factor γ
Compare Q-learning with γ=0.5 (myopic, cares only about immediate reward) vs γ=0.99 (far-sighted, cares about long-term reward). How does policy differ?


### Exercise 4: SARSA vs Q-Learning
SARSA is on-policy Q-learning (uses the action actually taken, not the greedy action). Implement SARSA and compare sample efficiency with Q-learning on the grid world.


## 6. Summary

| Concept | Skillsoft Course | Notebook |
|---------|-----------------|----------|
| RL framework (agent, env, reward) | RL Essentials (29m) | 07_reinforcement_learning.ipynb |
| MDP formulation | Covered in Essentials | 07_reinforcement_learning.ipynb |
| Q-learning algorithm | RL Tools & Frameworks (34m) | 07_reinforcement_learning.ipynb |
| ε-greedy policy | Covered in Tools & Frameworks | 07_reinforcement_learning.ipynb |
| OpenAI Gym | Tools & Frameworks installation | 07_reinforcement_learning.ipynb |

Key Takeaway: Reinforcement learning gives agents the ability to learn decision-making policies through trial-and-error interaction with an environment — a core capability for autonomous ML systems as an Architect.